# ACT Training on comma2k19

git clone https://github.com/Arivoli-A/comma2k19_FSD.git -b Arivoli

In [1]:
!pip install lerobot
from pathlib import Path

PROJECT_DIR = Path.cwd()
CHUNK_PATH = PROJECT_DIR / "comma2k19_data" / "extracted" / "Chunk_1"
DATASET_REPO_ID = "local/comma2k19_act"
DATASET_ROOT = PROJECT_DIR / "lerobot_datasets" / DATASET_REPO_ID
OUTPUT_DIR = PROJECT_DIR / "outputs" / "train" / "act_comma2k19"

print("Project:", PROJECT_DIR)
print("Chunk:", CHUNK_PATH)
print("Dataset root:", DATASET_ROOT)
print("Output:", OUTPUT_DIR)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 81.1 MB/s eta 0:00:00
   

Project: /content
Chunk: /content/comma2k19_data/extracted/Chunk_1
Dataset root: /content/lerobot_datasets/local/comma2k19_act
Output: /content/outputs/train/act_comma2k19


In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path(".").resolve()))

from types import SimpleNamespace

import torch
import matplotlib.pyplot as plt
import numpy as np

import comma2k19_FSD.act_training as act_training

In [2]:
args = SimpleNamespace(

    # --- paths ---
    base_dir=Path("."),
    dataset_root=Path("./lerobot_datasets"),
    repo_id="local/comma2k19_act",
    output_dir=Path("./outputs/train/act_comma2k19"),
    project_scripts_dir=Path("./comma2k19_FSD"),

    # --- chunks ---
    chunks=[1, 2],
    hf_token=None,

    # --- conversion ---
    width=256,
    height=256,
    fps=20,
    future_time=1.0,
    max_episodes=None,
    max_frames=None,

    # --- windowing ---
    obs_horizon=1,
    action_horizon=16,
    pred_horizon=16,

    # --- training ---
    device="cuda",
    batch_size=8,
    steps=10000,
    num_workers=4,
    val_fraction=0.1,
    log_every=100,
    save_every=1000,
    seed=42,

    # --- inference ---
    checkpoint=None,
    inference_episodes=3,

    # --- skip flags ---
    skip_download=False,
    skip_extract=False,
    skip_convert=False,
    skip_train=False,
)

# Load Dataset

In [ ]:
act_training.stage_download_extract(args)
act_training.stage_convert(args)

dataset = act_training.load_lerobot_dataset(args)

print(dataset)
print(dataset.features)


  Stage 1 — Chunk 1: download + extract

  Stage 1 — Chunk 2: download + extract

  Stage 2 — Chunk 1: convert to LeRobot


In [ ]:
sample = dataset[0]

for k, v in sample.items():
    if hasattr(v, "shape"):
        print(k, v.shape)
    else:
        print(k, type(v))

# Sliding window observations and action chunk

In [ ]:
sample = dataset[100]

images = sample["observation.images.front"]

print(images.shape)

fig, axes = plt.subplots(1, len(images), figsize=(15, 4))

for i in range(len(images)):
    axes[i].imshow(images[i])
    axes[i].set_title(f"obs {i}")
    axes[i].axis("off")

plt.show()

In [ ]:
sample = dataset[100]

actions = sample["action"]

print(actions.shape)
print(actions)

In [ ]:
plt.plot(actions[:, 0])
plt.title("Future speed trajectory")
plt.xlabel("Future step")
plt.ylabel("Speed")
plt.grid()
plt.show()

plt.plot(actions[:, 1])
plt.title("Future steering trajectory")
plt.xlabel("Future step")
plt.ylabel("Steer")
plt.grid()
plt.show()

# Train/Val split

In [ ]:
train_set, val_set, val_episode_ids = act_training.split_dataset(
    dataset,
    args.val_fraction,
    args.seed,
)

train_loader, val_loader = act_training.make_dataloaders(
    train_set,
    val_set,
    args,
)

print(len(train_set))
print(len(val_set))

#Build ACT model

In [ ]:
policy, device = act_training.make_act_policy(
    dataset,
    args,
)

print(type(policy))

In [ ]:
batch = next(iter(train_loader))

batch = {
    k: v.to(device)
    if hasattr(v, "to")
    else v
    for k, v in batch.items()
}

In [ ]:
loss, info = policy.forward(batch)

print("loss =", loss.item())

print(info)

In [ ]:
with torch.no_grad():
    out = policy.model(batch)

print(type(out))

if isinstance(out, dict):
    print(out.keys())

#Training

In [ ]:
# Train
act_training.train(
    policy,
    train_loader,
    val_loader,
    args,
    device,
)

#Evaluation

In [ ]:
ckpt_dir = Path("./outputs/train/act_comma2k19/checkpoints/last/pretrained_model")

policy = act_training.load_policy_from_checkpoint(ckpt_dir, device)
policy.eval()

In [ ]:
val_loss = act_training.evaluate(
    policy,
    val_loader,
    device,
    max_batches=50,
)

print("Validation loss:", val_loss)

## Full rollout evaluation

In [ ]:
import torch
import numpy as np

all_mae_speed = []
all_mae_steer = []

policy.eval()

with torch.no_grad():
    for batch in val_loader:

        batch = {
            k: v.to(device) if hasattr(v, "to") else v
            for k, v in batch.items()
        }

        pred = policy.select_action(batch)   # (B, H, 2)
        gt   = batch["action"]               # (B, H, 2)

        mae_speed = torch.abs(pred[..., 0] - gt[..., 0]).mean().item()
        mae_steer = torch.abs(pred[..., 1] - gt[..., 1]).mean().item()

        all_mae_speed.append(mae_speed)
        all_mae_steer.append(mae_steer)

print("Chunk MAE speed:", np.mean(all_mae_speed))
print("Chunk MAE steer:", np.mean(all_mae_steer))

## Horizon-wise error

In [ ]:
import torch
import numpy as np

H = args.action_horizon

errors_speed = np.zeros(H)
errors_steer = np.zeros(H)
counts = np.zeros(H)

policy.eval()

with torch.no_grad():
    for batch in val_loader:

        batch = {
            k: v.to(device) if hasattr(v, "to") else v
            for k, v in batch.items()
        }

        pred = policy.select_action(batch)  # (B, H, 2)
        gt   = batch["action"]              # (B, H, 2)

        for t in range(H):
            errors_speed[t] += torch.abs(pred[:, t, 0] - gt[:, t, 0]).sum().item()
            errors_steer[t] += torch.abs(pred[:, t, 1] - gt[:, t, 1]).sum().item()
            counts[t] += pred.shape[0]

errors_speed /= counts
errors_steer /= counts

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(errors_speed, label="speed MAE")
plt.plot(errors_steer, label="steer MAE")
plt.xlabel("future step")
plt.ylabel("MAE")
plt.title("Horizon-wise ACT error")
plt.legend()
plt.grid()
plt.show()

## Rollout simulation (closed-loop style)

In [ ]:
policy.eval()

ep_idx = val_episode_ids[0]

start = int(dataset.episode_data_index["from"][ep_idx])
end   = int(dataset.episode_data_index["to"][ep_idx])

errors = []

with torch.no_grad():
    for i in range(start, min(start + 200, end)):

        sample = dataset[i]

        obs = {
            "observation.images.front": sample["observation.images.front"].unsqueeze(0).to(device),
            "observation.state": sample["observation.state"].unsqueeze(0).to(device),
        }

        pred_chunk = policy.select_action(obs)[0]  # (H,2)

        pred = pred_chunk[0]
        gt   = sample["action"]

        error = torch.abs(pred - gt.to(device)).mean().item()
        errors.append(error)

print("Rollout MAE:", np.mean(errors))

## Visualize predicted trajectory vs ground truth

In [ ]:
batch = next(iter(val_loader))

batch = {
    k: v.to(device) if hasattr(v, "to") else v
    for k, v in batch.items()
}

pred = policy.select_action(batch)
gt   = batch["action"]

i = 0  # first sample in batch

plt.figure(figsize=(10,4))

plt.plot(gt[i,:,0].cpu(), label="GT speed")
plt.plot(pred[i,:,0].cpu(), label="Pred speed")
plt.legend()
plt.title("Speed trajectory")
plt.grid()
plt.show()

plt.figure(figsize=(10,4))

plt.plot(gt[i,:,1].cpu(), label="GT steer")
plt.plot(pred[i,:,1].cpu(), label="Pred steer")
plt.legend()
plt.title("Steering trajectory")
plt.grid()
plt.show()